# Fine-Tuning GPT-4.1-mini for Named Entity Recognition (NER)
---

In [2]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import json
from openai import OpenAI

### Initialise

In [4]:

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


###  Upload Training File

In [6]:
# Make sure the JSONL file is formatted according to OpenAI's fine-tuning schema
size = 1500
train_file_path = f"output_data/finetuned_data/train.jsonl"
val_file_path = f"output_data/finetuned_data/val.jsonl"
test_file_path = f"output_data/finetuned_data/test.jsonl"
train_file = client.files.create(
    file=open(train_file_path, "rb"),
    purpose="fine-tune"
)
val_file = client.files.create(
    file=open(val_file_path, "rb"),
    purpose="fine-tune"
)
training_file_id = train_file.id
val_file_id = val_file.id
print("Training file uploaded.")
print("Training File ID:", training_file_id)
print("Validation File ID:", val_file_id)


Training file uploaded.
Training File ID: file-DLJbJV8nEpNJ1NnoNvmECM
Validation File ID: file-6yVLLwwiyME3ACypTJ4HKB


### Create Fine-Tuning Job

In [7]:
fine_tune_response = client.fine_tuning.jobs.create(
    training_file=training_file_id,
     validation_file=val_file_id,
    model= "gpt-4.1-mini-2025-04-14" #gpt-3.5-turbo #gpt-4o-mini-2024-07-18 #gpt-4.1-mini-2025-04-14
)

job_id = fine_tune_response.id
print("Fine-tuning job started.")
print("Job ID:", job_id)


Fine-tuning job started.
Job ID: ftjob-m80aY0hp4Z2d6TOLD9zrxVwd


### Retrieve Fine-Tuning Job Info

In [8]:
job = client.fine_tuning.jobs.retrieve(job_id) #"ftjob-fmyPUoUMsUp33drLyKY8ERdQ"
print("📄 Fine-tuning job status:")
print(job)


📄 Fine-tuning job status:
FineTuningJob(id='ftjob-m80aY0hp4Z2d6TOLD9zrxVwd', created_at=1764973058, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-Naf8JqIqj78cDx1u4sa0mO5I', result_files=[], seed=1441106118, status='validating_files', trained_tokens=None, training_file='file-DLJbJV8nEpNJ1NnoNvmECM', validation_file='file-6yVLLwwiyME3ACypTJ4HKB', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)


### Generate Predictions from the Fine-Tuned Model

In [ ]:
# (Note: Ensure the job is completed and `fine_tuned_model` is available before running this)

import time

# ⏳ Wait or poll until fine-tuning job is complete
while job.fine_tuned_model is None:
    print("⏳ Waiting for fine-tuned model to be ready...")
    time.sleep(10)
    job = client.fine_tuning.jobs.retrieve(job_id)

model_id = job.fine_tuned_model
print("✅ Fine-tuned model is ready:", model_id)


⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...
⏳ Waiting for fine-tuned model to be ready...


In [ ]:
client.fine_tuning.jobs.retrieve(job_id)

### Save Predictions

In [ ]:
import json
def generate_predictions(input_file, output_file):
    count = 0

    with open(input_file, 'r', encoding='utf-8') as f:
        examples = [json.loads(line) for line in f]

    with open(output_file, 'w', encoding='utf-8') as out_f:
        for ex in examples:
            user_input = ex["messages"][1]["content"]
            system_prompt = ex["messages"][0]["content"]

            # Try to extract expected output, default to {}
            try:
                expected_output = json.loads(ex["messages"][2]["content"])
            except json.JSONDecodeError:
                expected_output = {}
                
            
            # Initialize predicted_output and raw_content
            predicted_output = None
            raw_content = None

            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                temperature=0
            )

            raw_content = response.choices[0].message.content
            try:
                predicted_output = json.loads(raw_content)
            except json.JSONDecodeError:
                predicted_output = None  # Keep as None, will store raw content instead

            result = {
                "input": user_input,
                "predicted": predicted_output,
                "raw_response": raw_content,
                "expected": expected_output,
            }

            out_f.write(json.dumps(result) + "\n")
            count += 1

    print(f"\n✅ {count} predictions saved to: {output_file}")



In [ ]:

output_predictions_path = f"output_gpt4.1_ft.jsonl"

generate_predictions(test_file_path, output_predictions_path)

### Run Prediction Generation

### Evaluate predictions

In [10]:
# I have to guess
# import pandas as pd
# from collections import defaultdict
# def evaluate_predictions(output_predictions_path, labels = {'organisations', 'grants', 'authors'}):
#     final_scores = defaultdict(lambda: defaultdict(int))
    

#     # Global totals for micro-averaging
#     total_tp = total_fp = total_fn = 0
#     f1_scores = []
#     count = 0

#     with open(output_predictions_path) as infile:
#         for row in map(json.loads, infile):
            
#             preds = defaultdict(list)
#             gold = defaultdict(list)
#             for p in row.get('predicted', []) or []:
#                 preds[p.lower()] = row['predicted'][p]
#             for e in row['expected']:
#                 gold[e.lower()] = row['expected'][e]

#             for l in labels:
#                 pred_set = set(preds[l])
#                 gold_set = set(gold[l])
#                 tp = len(pred_set & gold_set)
#                 fp = len(pred_set - gold_set)
#                 fn = len(gold_set - pred_set)
#                 final_scores[l]['tp'] += tp
#                 final_scores[l]['fp'] += fp
#                 final_scores[l]['fn'] += fn

#                 total_tp += tp
#                 total_fp += fp
#                 total_fn += fn
#             count +=1
#     # Compute per-label metrics
#     for ent_type, counts in final_scores.items():
#         tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
#         precision = tp / (tp + fp) if tp + fp else 0.0
#         recall = tp / (tp + fn) if tp + fn else 0.0
#         f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

#         final_scores[ent_type].update({
#             'precision': round(precision, 4),
#             'recall': round(recall, 4),
#             'f1': round(f1, 4)
#         })

#         f1_scores.append(f1)

#     # Micro-averaged
#     micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
#     micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
#     micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) else 0.0

#     # Macro-averaged
#     macro_precision = sum(final_scores[l]['precision'] for l in labels) / len(labels)
#     macro_recall = sum(final_scores[l]['recall'] for l in labels) / len(labels)
#     macro_f1 = sum(f1_scores) / len(labels)

#     final_scores['micro'] = {
#         'tp': total_tp,
#         'fp': total_fp,
#         'fn': total_fn,
#         'precision': round(micro_precision, 4),
#         'recall': round(micro_recall, 4),
#         'f1': round(micro_f1, 4)
#     }
#     final_scores['macro'] = {
#         'precision': round(macro_precision, 4),
#         'recall': round(macro_recall, 4),
#         'f1': round(macro_f1, 4)
#     }

#     return final_scores
# pd.DataFrame(evaluate_predictions(output_predictions_path))
# # pd.DataFrame(evaluate_predictions("../data/annotated/20250530_results/ft_gpt41_results.jsonl"))

,grants,authors,organisations,micro,macro
tp,2734.0000,2587.0000,3489.0000,8810.0000,NaN
fp,188.0000,260.0000,1124.0000,1572.0000,NaN
fn,588.0000,1071.0000,2127.0000,3786.0000,NaN
precision,0.9357,0.9087,0.7563,0.8486,0.8669
recall,0.8230,0.7072,0.6213,0.6994,0.7172
f1,0.8757,0.7954,0.6822,0.7668,0.7844


In [11]:
import pandas as pd
from collections import defaultdict
output_predictions_path = "../data/annotated/1000/test_finetuned_gpt.jsonl"
def evaluate_predictions(output_predictions_path, labels = {'organisations', 'grants', 'authors'}):
    final_scores = defaultdict(lambda: defaultdict(int))
    

    # Global totals for micro-averaging
    total_tp = total_fp = total_fn = 0
    f1_scores = []
    count = 0

    with open(output_predictions_path) as infile:
        for row in map(json.loads, infile):
            
            preds = defaultdict(list)
            gold = defaultdict(list)
            for p in row.get('predicted', []) or []:
                preds[p.lower()] = row['predicted'][p]
            for e in row['expected']:
                gold[e.lower()] = row['expected'][e]

            for l in labels:
                pred_set = set(preds[l])
                gold_set = set(gold[l])
                tp = len(pred_set & gold_set)
                fp = len(pred_set - gold_set)
                fn = len(gold_set - pred_set)
                final_scores[l]['tp'] += tp
                final_scores[l]['fp'] += fp
                final_scores[l]['fn'] += fn

                total_tp += tp
                total_fp += fp
                total_fn += fn
            count +=1
    # Compute per-label metrics
    for ent_type, counts in final_scores.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        final_scores[ent_type].update({
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4)
        })

        f1_scores.append(f1)

    # Micro-averaged
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) else 0.0

    # Macro-averaged
    macro_precision = sum(final_scores[l]['precision'] for l in labels) / len(labels)
    macro_recall = sum(final_scores[l]['recall'] for l in labels) / len(labels)
    macro_f1 = sum(f1_scores) / len(labels)

    final_scores['micro'] = {
        'tp': total_tp,
        'fp': total_fp,
        'fn': total_fn,
        'precision': round(micro_precision, 4),
        'recall': round(micro_recall, 4),
        'f1': round(micro_f1, 4)
    }
    final_scores['macro'] = {
        'precision': round(macro_precision, 4),
        'recall': round(macro_recall, 4),
        'f1': round(macro_f1, 4)
    }

    return final_scores
pd.DataFrame(evaluate_predictions(output_predictions_path))
# pd.DataFrame(evaluate_predictions("../data/annotated/20250530_results/ft_gpt41_results.jsonl"))

KeyError: 'expected'

In [ ]:
input_file = '../data/raw/20250530/raw_text_1_finetuned.jsonl'
with open(input_file, 'r', encoding='utf-8') as f:
    
    examples = [json.loads(line) for line in f]

In [39]:
examples[0]

{'messages': [{'role': 'system',
   'content': 'Extract all organisations, authors, and grant entities from the following text.'},
  {'role': 'user',
   'content': 'The authors thank Peter Burbelo and Sarthak Gupta for their critical reading and suggestions on the manuscript, Sharon D. Adams for performing HLA allele typing and Kenichiro Tanabe for advice on statistical analysis. Members of the Childhood Myositis Heterogeneity Study Group who contributed to this project: Bita Arabshahi, April Bingham, Victoria Cartwright, Rodolfo Curiel, Marietta M. DeGuzman, Barbara Anne Eberhard, Barbara S. Edelheit, Terri H. Finkel, William Hannan, Michael Henrickson, Adam M. Huber, Anna Jansen, Steven J. Klein, Bianca Lang, Carol B. Lindsley, Gulnara Mamyrova, Frederick W. Miller, Stephen R. Mitchell, Kabita Nanda, Payam Noroozi Farhadi, Murray H. Passo, Donald A. Person, Tova Ronis, Adam Schiffenbauer, Bracha Shaham, Matthew L. Stoll, Sangeeta H. Sule, Ira N. Targoff, Scott A. Vogelgesang, Rita Vo

In [40]:
input_file = '../data/raw/20250530/raw_text_1.jsonl'
with open(input_file, 'r', encoding='utf-8') as f:
    
    examples = [json.loads(line) for line in f]

In [41]:
text_dict = {}
for ex in examples:
    text_dict[ex['text']] = (ex['id'], ex['section'])

In [42]:
with open('finetune_withdata1.jsonl', 'w') as outfile:
    with open('finetune_tagged.jsonl') as infile:
        for row in map(json.loads, infile):
            if row['input'] in text_dict:
                pub_id, section = text_dict[row['input']]
                outfile.write(json.dumps(
                    {'id': pub_id,
                    'section': section,
                    'text': row['input'],
                    'entities': row['predicted']}
                )+'\n')
            else:
                print(row['input'], "doesn't exist")    

,grants,authors,organisations,micro,macro
tp,1030.0000,975.0000,1834.0000,3839.0000,NaN
fp,168.0000,275.0000,598.0000,1041.0000,NaN
fn,140.0000,250.0000,494.0000,884.0000,NaN
precision,0.8598,0.7800,0.7541,0.7867,0.7980
recall,0.8803,0.7959,0.7878,0.8128,0.8213
f1,0.8699,0.7879,0.7706,0.7995,0.8095


In [64]:
pd.DataFrame(evaluate_predictions('finetune_predictions.jsonl'))

,grants,authors,organisations,micro,macro
tp,1202.0000,1039.0000,2008.0000,4249.0000,NaN
fp,157.0000,275.0000,559.0000,991.0000,NaN
fn,116.0000,274.0000,471.0000,861.0000,NaN
precision,0.8845,0.7907,0.7822,0.8109,0.8191
recall,0.9120,0.7913,0.8100,0.8315,0.8378
f1,0.8980,0.7910,0.7959,0.8211,0.8283


In [74]:
with open("finetune_predictions_test.jsonl") as infile:
    for row in map(json.loads, infile):
        print(row.keys())
        

dict_keys(['input', 'predicted', 'expected'])
dict_keys(['input', 'predicted', 'expected'])
dict_keys(['input', 'predicted', 'expected'])
dict_keys(['input', 'predicted', 'expected'])
